# Pathumma Fastcon + LLM spelling correction

**DSI443 TU Speech Recognition Challenge 2026 · experiment E09**

Idea: the ASR model often writes a Thai word with one wrong letter (e.g. `แรงเสียดทาน` → `แรงเสียทาน`).
newmm then cuts the non-word into several pieces, so **one wrong letter costs ~3 word errors**.
An LLM knows which Thai words exist, so it can repair them.

Pipeline:
1. **Part 1, ASR:** zero-shot `nectec/Pathumma-stt-th-fastconformer-rnnt-large` transcribes dev (240 clips) and test (186).
2. **Part 2, LLM:** a Thai LLM lists misspelled words per sentence. We apply a fix **only if it is a small edit**
   (≤ 2 characters), so the LLM can't rewrite sentences, invent words or change names and numbers.
3. Measure dev WER before/after, and write a Kaggle submission from the corrected test transcripts.

**Runtime:** GPU (T4). Part 1 ≈ 10 min (mostly installing NeMo). Part 2 ≈ 15–30 min.

> Rules: only open-source models run here, no paid API. Test transcripts are never looked at or tuned on.

## Part 0: settings and data
Kaggle API token: add it in Colab **Secrets** (key icon on the left) as `KAGGLE_API_TOKEN`,
or upload your `kaggle.json` when asked. You must have joined the competition.

In [ ]:
import os, glob, json, re, subprocess, sys

COMPETITION = "dsi-443-tu-speech-recognition-challenge-2026"
DATA_DIR = "/content/data"
WORK_DIR = "/content/work"          # survives "Restart session" (not "Disconnect and delete runtime")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

ASR_MODEL = "nectec/Pathumma-stt-th-fastconformer-rnnt-large"

In [ ]:
# Kaggle credentials: Colab secret KAGGLE_API_TOKEN, else upload kaggle.json
try:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    print("Using KAGGLE_API_TOKEN from Colab secrets")
except Exception:
    from google.colab import files
    print("Upload kaggle.json")
    uploaded = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    with open("/root/.kaggle/kaggle.json", "wb") as f:
        f.write(uploaded["kaggle.json"])
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

!pip install -q -U kaggle
!kaggle competitions download -c {COMPETITION} -p {DATA_DIR} -q
!cd {DATA_DIR} && unzip -q -o "*.zip" && rm -f *.zip
CSV_DIR = os.path.dirname(glob.glob(f"{DATA_DIR}/**/dev.csv", recursive=True)[0])
print("CSV folder:", CSV_DIR)

## Part 1: ASR with zero-shot Pathumma Fastcon

In [ ]:
!pip install -q "nemo_toolkit[asr]" pythainlp

In [ ]:
import librosa, soundfile as sf, pandas as pd, torch

SR = 16_000
mp3_index = {os.path.basename(p): p for p in glob.glob(f"{DATA_DIR}/**/*.mp3", recursive=True)}

def to_wav(audio, wav_dir):
    # decode one MP3 to 16 kHz mono WAV (the format Pathumma was trained on)
    src = os.path.join(CSV_DIR, audio)
    src = src if os.path.exists(src) else mp3_index[os.path.basename(audio)]
    wav = os.path.join(wav_dir, os.path.splitext(os.path.basename(audio))[0] + ".wav")
    y, _ = librosa.load(src, sr=SR, mono=True)
    sf.write(wav, y, SR)
    return wav

splits = {}
for split in ("dev", "test"):
    df = pd.read_csv(os.path.join(CSV_DIR, f"{split}.csv"))
    wav_dir = f"/content/wav/{split}"
    os.makedirs(wav_dir, exist_ok=True)
    df["wav"] = [to_wav(a, wav_dir) for a in df["audio"]]
    splits[split] = df
    print(split, len(df), "clips")

In [ ]:
import nemo.collections.asr as nemo_asr
from huggingface_hub import HfApi, hf_hub_download

nemo_file = next(f for f in HfApi().list_repo_files(ASR_MODEL) if f.endswith(".nemo"))
asr = nemo_asr.models.ASRModel.restore_from(hf_hub_download(ASR_MODEL, nemo_file), map_location="cuda")
asr.eval()

def transcribe(wavs):
    out = asr.transcribe(wavs, batch_size=16, verbose=False)
    if isinstance(out, tuple):  # older NeMo RNNT returns (best, all_hypotheses)
        out = out[0]
    return [(o if isinstance(o, str) else o.text).strip() for o in out]

dev, test = splits["dev"], splits["test"]
dev["hyp"] = transcribe(dev["wav"].tolist())
test["hyp"] = transcribe(test["wav"].tolist())
dev[["audio", "sentence", "hyp"]].rename(columns={"sentence": "ref"}).to_csv(f"{WORK_DIR}/asr_dev.csv", index=False)
test[["audio", "hyp"]].to_csv(f"{WORK_DIR}/asr_test.csv", index=False)
print("saved ASR transcripts to", WORK_DIR)

## ⚠️ Restart before Part 2
NeMo installs its own `transformers` version, which is too old for the Gemma 3 LLM.
Do **Runtime → Restart session**, then run Part 2 from the next cell. Files in `/content/work` are kept.

## Part 2: LLM spelling correction

In [ ]:
!pip install -q -U transformers accelerate pythainlp

In [ ]:
import os, re, json, math, time
import pandas as pd, torch
from pythainlp.tokenize import word_tokenize
from transformers import AutoTokenizer, AutoModelForCausalLM

WORK_DIR = "/content/work"

# --- settings ------------------------------------------------------------
LLM_ID = "typhoon-ai/typhoon2.1-gemma3-4b"   # Thai LLM (Gemma 3). Alternative: "Qwen/Qwen3-4B-Instruct-2507"
# Gemma 3 overflows in float16, so it needs bfloat16 (emulated on a T4: slower but works).
LLM_DTYPE = torch.bfloat16 if "gemma" in LLM_ID else torch.float16
MAX_EDIT = 2            # accept a fix only if it changes at most this many characters
MAX_EDIT_RATIO = 0.34   # ...and at most this share of the word's length
# -------------------------------------------------------------------------

dev = pd.read_csv(f"{WORK_DIR}/asr_dev.csv").fillna("")
test = pd.read_csv(f"{WORK_DIR}/asr_test.csv").fillna("")
print(len(dev), "dev /", len(test), "test transcripts loaded")

In [ ]:
# WER exactly like the competition: newmm words, edits / reference words over the whole set
def words(text):
    return [w for w in word_tokenize(str(text), engine="newmm", keep_whitespace=False) if w.strip()]

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, y in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y))
        prev = cur
    return prev[-1]

def wer(refs, hyps):
    e = n = 0
    for r, h in zip(refs, hyps):
        rw = words(r)
        e += edit_distance(rw, words(h))
        n += len(rw)
    return e / max(n, 1)

def cer(refs, hyps):
    strip = lambda s: re.sub(r"\s+", "", str(s))
    e = n = 0
    for r, h in zip(refs, hyps):
        e += edit_distance(list(strip(r)), list(strip(h)))
        n += len(strip(r))
    return e / max(n, 1)

print(f"zero-shot Pathumma on dev: WER {wer(dev['ref'], dev['hyp']):.4f}  CER {cer(dev['ref'], dev['hyp']):.4f}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=LLM_DTYPE, device_map="cuda")
llm.eval()
print("loaded", LLM_ID)

In [ ]:
# Few-shot examples are made up for this prompt (not taken from dev or test).
INSTRUCTIONS = (
    "You check Thai speech-recognition output for spelling mistakes.\n"
    "Find words that are misspelled or are not real Thai words, and give the correct spelling.\n"
    "Rules:\n"
    "- Only fix misspelled or non-existent words. Do not rephrase, add or remove words.\n"
    "- Do not change English words, names written in English, numbers or punctuation.\n"
    "- If nothing is wrong, answer [].\n"
    'Answer with JSON only: [{"wrong": "...", "correct": "..."}]'
)
EXAMPLES = [
    ("เมื่อวานฉันไปซื้อผลไม้ที่ตลาดและได้ทุเรีนมาสองลูก", [{"wrong": "ทุเรีน", "correct": "ทุเรียน"}]),
    ("การประชุมครั้งนี้มีผู้เข้าร่วมจากหลายประเทศในเอเชีย", []),
    ("นักดาราศาสตร์ศึกษาการเคลื่อนที่ของดาวเคาระห์รอบดวงอาทิตย์", [{"wrong": "ดาวเคาระห์", "correct": "ดาวเคราะห์"}]),
]

def build_messages(sentence):
    messages = [{"role": "system", "content": INSTRUCTIONS}]
    for text, fixes in EXAMPLES:
        messages.append({"role": "user", "content": text})
        messages.append({"role": "assistant", "content": json.dumps(fixes, ensure_ascii=False)})
    messages.append({"role": "user", "content": sentence})
    return messages

@torch.no_grad()
def ask_llm(sentence):
    ids = tokenizer.apply_chat_template(build_messages(sentence), add_generation_prompt=True,
                                        return_tensors="pt").to(llm.device)
    out = llm.generate(ids, max_new_tokens=200, do_sample=False)
    reply = tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    match = re.search(r"\[.*\]", reply, re.S)
    try:
        fixes = json.loads(match.group(0)) if match else []
    except json.JSONDecodeError:
        fixes = []
    return reply, [f for f in fixes if isinstance(f, dict)]

In [ ]:
LATIN_OR_DIGIT = re.compile(r"[A-Za-z0-9]")

def accept(fix, sentence):
    # keep only small, safe edits; returns (ok, reason)
    wrong, correct = str(fix.get("wrong", "")).strip(), str(fix.get("correct", "")).strip()
    if not wrong or not correct or wrong == correct:
        return False, "empty or unchanged"
    if wrong not in sentence:
        return False, "wrong word not in sentence"
    if LATIN_OR_DIGIT.search(wrong + correct):
        return False, "touches English/number"
    d = edit_distance(list(wrong), list(correct))
    if d > MAX_EDIT or d > math.ceil(MAX_EDIT_RATIO * len(wrong)):
        return False, f"edit too big ({d} chars)"
    return True, f"{d} chars"

def correct_sentence(sentence):
    reply, fixes = ask_llm(sentence)
    fixed, log = sentence, []
    for f in fixes:
        ok, why = accept(f, fixed)
        log.append({"wrong": f.get("wrong"), "correct": f.get("correct"), "accepted": ok, "why": why})
        if ok:
            fixed = fixed.replace(str(f["wrong"]).strip(), str(f["correct"]).strip(), 1)
    return fixed, log, reply

# quick look at 3 dev sentences before running everything
for s in dev["hyp"].head(3):
    fixed, log, reply = correct_sentence(s)
    print(s, "\n  LLM:", reply.strip()[:200], "\n  ->", fixed, "\n")

In [ ]:
def run(df, name):
    t0 = time.time()
    fixed_all, logs = [], []
    for i, s in enumerate(df["hyp"]):
        fixed, log, reply = correct_sentence(s)
        fixed_all.append(fixed)
        logs.append({"audio": df["audio"].iloc[i], "fixes": log, "reply": reply})
        if (i + 1) % 25 == 0:
            print(f"{name}: {i + 1}/{len(df)}  ({(time.time() - t0) / 60:.1f} min)")
    json.dump(logs, open(f"{WORK_DIR}/llm_log_{name}.json", "w"), ensure_ascii=False, indent=1)
    return fixed_all

dev["fixed"] = run(dev, "dev")
dev.to_csv(f"{WORK_DIR}/llm_dev.csv", index=False)

In [ ]:
before_w, after_w = wer(dev["ref"], dev["hyp"]), wer(dev["ref"], dev["fixed"])
before_c, after_c = cer(dev["ref"], dev["hyp"]), cer(dev["ref"], dev["fixed"])
print(f"dev WER  {before_w:.4f} -> {after_w:.4f}  ({after_w - before_w:+.4f})")
print(f"dev CER  {before_c:.4f} -> {after_c:.4f}  ({after_c - before_c:+.4f})")

# which changed sentences got better or worse
changed = dev[dev["hyp"] != dev["fixed"]].copy()
changed["delta"] = [edit_distance(words(r), words(f)) - edit_distance(words(r), words(h))
                    for r, h, f in zip(changed["ref"], changed["hyp"], changed["fixed"])]
print(f"\n{len(changed)} sentences changed: {(changed.delta < 0).sum()} better, "
      f"{(changed.delta > 0).sum()} worse, {(changed.delta == 0).sum()} same")
for _, r in changed.sort_values("delta").iterrows():
    print(f"\n[{r.delta:+d} errors] {r.audio}\n  REF:   {r.ref}\n  ASR:   {r.hyp}\n  FIXED: {r.fixed}")

## Part 3: test predictions and submission
Only worth submitting if dev WER went down above.

In [ ]:
test["fixed"] = run(test, "test")

def segment(text):
    return " ".join(words(text))

import glob
order = pd.read_csv(glob.glob("/content/data/**/test.csv", recursive=True)[0])["audio"].tolist()
by_audio = dict(zip(test["audio"], test["fixed"]))
assert set(order) == set(by_audio), "test clips don't match test.csv"
sub = pd.DataFrame({"audio": order, "sentence": [segment(by_audio[a]) for a in order]})
sub.to_csv(f"{WORK_DIR}/submission_pathumma_llm.csv", index=False)
print(sub.head())

from google.colab import files
files.download(f"{WORK_DIR}/submission_pathumma_llm.csv")

## Results to record (experiment log E09)
- Dev WER / CER before and after the LLM, from the cell above.
- How many sentences changed, and how many got better or worse.
- Kaggle public score of `submission_pathumma_llm.csv`, compared with zero-shot (0.26) and B (0.24).